# Phase 8 — CLI Self-Update System Design

## Part 1 — System Overview

### Objective

The objective of this phase is to design a secure, reliable, and maintainable self-update mechanism for a Command Line Interface (CLI) application. Instead of implementing the update system in code, this phase focuses on designing the complete workflow, explaining every update step, identifying possible failure scenarios, and documenting how the application can safely update itself from GitHub.

The final design should be understandable, practical, and easy to defend during a technical discussion or interview.

## 1.1 What is a CLI Self-Update System?

A CLI Self-Update System is a mechanism that enables a command-line application to automatically detect, download, verify, and install newer versions of itself without requiring users to manually reinstall the software.

Instead of asking users to visit GitHub or reinstall the application every time a new version is released, the CLI application can check whether an update is available and perform the update safely.

A complete self-update system generally performs the following steps:

1. Detect the currently installed version.
2. Check GitHub for the latest available version.
3. Compare both versions.
4. Download the latest release if an update exists.
5. Verify the downloaded files.
6. Safely replace the old version.
7. Restart the application.
8. Roll back if the update fails.

The goal is to provide users with a simple, secure, and reliable update experience while protecting the application from corrupted downloads or failed installations.

## 1.2 Overall Architecture

The CLI self-update system follows a sequential workflow where the application checks for updates, validates downloaded files, installs the new version safely, and restores the previous version if any failure occurs.

### High-Level Workflow

User
↓
CLI Application
↓
Read Local Version
↓
Query GitHub Releases API
↓
Compare Versions
↓
Is Update Available?
├── No → Continue Running
└── Yes
        ↓
Download Release
        ↓
Verify Integrity
        ↓
Stage Update
        ↓
Replace Current Version
        ↓
Restart CLI
        ↓
Cleanup

## 1.3 XMind Structure

The complete system design will be organized using the following XMind structure.

CLI Self-Update System

├── Update Trigger
├── Version Management
├── Fetch Strategy
├── Integrity & Trust
├── Safe Update
├── Post Update
├── Rollback
├── Permissions
├── Distribution Channels
└── Deliverables

Each node will be expanded in the following sections with real commands, GitHub API endpoints, file locations, recovery mechanisms, and implementation details.

# Part 2 — Update Flow

This section describes how the CLI application detects, verifies, and downloads new software updates from GitHub. The update flow is designed to be simple, reliable, and secure while minimizing unnecessary network requests.

## 2.1 Update Trigger

The update process begins when the CLI decides to check for a newer version. Several triggering methods are possible depending on the application's design.

### Manual Trigger

The user explicitly requests an update.

Example:

mycli --update

The CLI immediately checks GitHub for the latest release.

---

### Startup Check

Whenever the CLI starts, it silently checks whether a newer version exists.

If an update is available, the CLI displays a notification such as:

"A newer version (v2.1.0) is available.
Run 'mycli --update' to install."

---

### Scheduled Check

The CLI stores the timestamp of the last update check.

Instead of checking GitHub every launch, it only performs a new check after a fixed interval (for example every 24 hours).

This reduces unnecessary API requests while keeping users informed about new releases.

## 2.2 Version Checking

After the update process is triggered, the CLI compares the currently installed version with the latest version available on GitHub.

Example:

Local Version:
v1.2.0

Latest GitHub Release:
v1.4.0

Since the versions are different, the CLI determines that an update is available.

If both versions are identical, no update is required and the application continues running normally.

Semantic Versioning (SemVer) should be used when comparing versions.

Format:

MAJOR.MINOR.PATCH

Example:

1.0.0
1.2.0
2.0.0

## 2.3 Version Source

The CLI requires two sources of version information.

### Local Version

The currently installed version is stored inside the application.

Possible locations include:

• src/mycli/__init__.py

Example:

__version__ = "1.2.0"

or

• pyproject.toml

The CLI reads this value during startup.

---

### Remote Version

The latest available version is fetched from GitHub Releases.

GitHub Releases API:

GET https://api.github.com/repos/<owner>/<repository>/releases/latest

The API returns metadata including:

- Latest version
- Release notes
- Download URLs
- Published date

The CLI compares the local version with the latest release before deciding whether to update.

## 2.4 Fetch Strategy

If a newer version is available, the CLI downloads the update.

Several update strategies are possible.

### Option 1 — GitHub Release Asset (Recommended)

Download the latest release asset directly from GitHub Releases.

Examples:

- Windows (.exe)
- Linux binary
- macOS binary
- Python wheel (.whl)

---

### Option 2 — pip Upgrade

Update the package using pip.

Example:

pip install --upgrade mycli

---

### Option 3 — Git Update

If the application was installed from source:

git pull

---

For this design, GitHub Release Assets are selected as the primary update method because they provide versioned releases, reliable download links, and official release metadata.

# Part 3 — Safe Update Design

This section explains how the CLI safely installs updates while protecting users from corrupted downloads, failed installations, and permission-related issues. A secure update process minimizes the risk of data loss and ensures that users can always recover if an update fails.

## 3.1 Integrity & Trust

Before installing any downloaded update, the CLI must verify that the update is authentic and has not been modified.

### Security Measures

• Download updates only over HTTPS.

• Accept updates only from the official GitHub repository.

Example:

https://github.com/<owner>/<repository>

• Verify the SHA-256 checksum of the downloaded file.

If the calculated checksum does not match the published checksum, the update must be rejected.

• Optionally verify a digital signature (GPG) for additional security.

These checks protect users from corrupted files, tampered downloads, and unofficial repositories.

## 3.2 Safe Apply

The update should never overwrite the current installation immediately.

Instead, the CLI follows a staged installation process.

### Safe Update Workflow

1. Download the update into a temporary directory.

Example:

Windows:

%TEMP%\mycli-update

Linux/macOS:

/tmp/mycli-update

2. Verify the downloaded files.

3. Extract or install the update into a staging directory.

4. Perform a basic validation test.

5. Replace the old installation only after all checks pass.

Using a temporary staging area prevents incomplete or failed downloads from damaging the existing installation.

## 3.3 Restart

After the update has been installed successfully, the CLI should restart itself.

The recommended workflow is:

1. Finish the installation.

2. Save any required configuration.

3. Start the updated version.

4. Exit the old process.

Additional post-update tasks may include:

• Clearing cache files.

• Running database migrations (if required).

• Displaying GitHub Release Notes.

Restarting ensures that users immediately begin using the latest version.

## 3.4 Rollback

If any step of the update process fails, the CLI must restore the previous working version.

Possible failure scenarios include:

• Network interruption during download.

• Corrupted download.

• Failed installation.

• Failed validation test.

• Permission errors.

### Recovery Strategy

1. Keep a backup of the current installation.

2. Abort the update if any verification fails.

3. Restore the previous version from the backup.

4. Inform the user that the update failed.

Rollback prevents users from being left with a broken or unusable application.

## 3.5 Permissions

Different installation methods require different permission levels.

### User Installation

Example:

pip install --user mycli

No administrator privileges are required.

---

### Virtual Environment

The update should occur inside the currently active virtual environment.

No system files should be modified.

---

### System-Wide Installation

Example:

/usr/local/bin/

or

C:\Program Files\

Administrator (Windows) or sudo (Linux/macOS) permissions may be required.

If sufficient permissions are not available, the CLI should stop the update and display a clear error message instead of attempting a partial installation.

# Part 4 — Distribution Channel Comparison

A CLI application can be distributed and updated through different channels. Each method has its own advantages, limitations, and suitable use cases. Choosing the correct distribution strategy depends on the target users, operating system, and deployment requirements.

## 4.1 Distribution Channel Comparison

| Distribution Method | Self-Update Supported | Best Use Case | Advantages | Limitations |
|---------------------|----------------------|---------------|------------|-------------|
| GitHub Releases | Yes | Standalone CLI applications | Easy release management, downloadable binaries, release notes | Requires GitHub access |
| pip | No | Python packages and libraries | Simple installation, Python ecosystem integration | Users must manually run `pip install --upgrade` |
| pipx | No | Python CLI applications | Isolated environments, avoids dependency conflicts | Manual upgrade using `pipx upgrade` |
| Homebrew (brew) | No | macOS applications | Easy package management for macOS users | Limited to macOS |
| npm | No | Node.js CLI tools | Well integrated with JavaScript ecosystem | Only suitable for Node.js applications |
| Built-in Self-Update | Yes | Professional standalone CLI tools | Fully automatic update experience, complete control over update process | More complex to design and maintain |

## 4.2 Recommended Distribution Strategy

For this project, GitHub Releases combined with a built-in self-update mechanism is the recommended approach.

### Reasons

- GitHub Releases provides official version management.
- Release assets are easy to download and organize.
- Release notes are available through the GitHub Releases API.
- Version comparison is straightforward using release tags.
- The application maintains full control over the update process.
- Security checks such as checksum verification can be integrated before installation.

This approach provides a reliable, secure, and user-friendly update experience for standalone CLI applications.

## 4.3 Summary

Different distribution channels are suitable for different software ecosystems.

- **GitHub Releases** is ideal for standalone CLI applications.
- **pip** is best for Python packages.
- **pipx** is recommended for isolated Python CLI tools.
- **Homebrew** is designed for macOS users.
- **npm** is intended for Node.js applications.
- **Built-in Self-Update** offers the best user experience for applications that require automatic update management and complete control over the update process.

The selected approach for this system design is a GitHub Releases–based self-update mechanism because it balances simplicity, security, maintainability, and user convenience.

# Part 5 — Final Deliverables

This section contains the final project deliverables required for the CLI Self-Update System Design assignment. It includes a step-by-step update runbook, a personal reflection, and submission checklists to ensure that all required artifacts are complete and ready for submission.

## 5.1 CLI Self-Update Runbook (cli-self-update-steps.md)

### Step 1 — Check Current Version

**What happens**

The CLI reads its currently installed version.

**Command / API**

Local version from:

- src/mycli/__init__.py
- pyproject.toml

**Possible Failure**

Version information cannot be read.

**Recovery**

Display an error message and stop the update process.

---

### Step 2 — Fetch Latest Release

**What happens**

The CLI requests the latest release information from GitHub.

**Command / API**

GET https://api.github.com/repos/<owner>/<repository>/releases/latest

**Possible Failure**

Network timeout or GitHub API unavailable.

**Recovery**

Retry the request or notify the user.

---

### Step 3 — Compare Versions

**What happens**

Compare the installed version with the latest GitHub release.

**Possible Failure**

Invalid version format.

**Recovery**

Abort the update and report the error.

---

### Step 4 — Download Update

**What happens**

Download the latest release asset into a temporary directory.

**Possible Failure**

Interrupted download.

**Recovery**

Retry the download or restart it.

---

### Step 5 — Verify Integrity

**What happens**

Validate the downloaded file using SHA-256 checksum.

**Possible Failure**

Checksum mismatch.

**Recovery**

Delete the file and download it again.

---

### Step 6 — Stage Update

**What happens**

Extract or install the update into a staging directory.

**Possible Failure**

Extraction or installation failure.

**Recovery**

Delete the staging directory and keep the current installation.

---

### Step 7 — Apply Update

**What happens**

Replace the old installation with the validated update.

**Possible Failure**

Permission denied or file lock.

**Recovery**

Request administrator permission or cancel the update safely.

---

### Step 8 — Restart CLI

**What happens**

Launch the updated version and close the old process.

**Possible Failure**

Restart failure.

**Recovery**

Notify the user and allow manual restart.

---

### Step 9 — Cleanup

**What happens**

Delete temporary files and update caches if necessary.

**Possible Failure**

Temporary files remain.

**Recovery**

Remove them during the next update cycle.

---

### Step 10 — Rollback

**What happens**

Restore the previous version if any update step fails.

**Possible Failure**

Backup not available.

**Recovery**

Inform the user and recommend reinstalling the latest stable release.

## 5.2 Reflection

During this assignment, I used AI as a research and learning assistant to understand different approaches for designing a secure CLI self-update system. Every major concept, including version checking, update flow, rollback, integrity verification, and permission handling, was reviewed and verified before being included in the final design.

After reviewing the information, I simplified several update steps to make the workflow easier to understand and explain during a technical discussion. Instead of including unnecessary complexity, I focused on a practical and maintainable design.

The topic I am currently least confident about is digital signature verification (GPG), as I have limited hands-on experience with implementing it in real-world CLI applications. I understand its purpose and benefits, but I would like to explore it further through practical implementation.

## 5.3 Final Review Checklist

Before submission, verify the following:

- Complete system overview prepared.
- Update flow documented.
- Version management explained.
- Fetch strategy documented.
- Integrity verification included.
- Safe update process described.
- Restart procedure documented.
- Rollback strategy explained.
- Permission model documented.
- Distribution comparison completed.
- Runbook completed.
- Reflection completed.
- XMind diagram reviewed.
- All documentation proofread.

## 5.4 XMind Export Checklist

Before exporting the XMind file:

- Root node is clearly labeled.
- All required sections are included.
- Every node contains meaningful details.
- GitHub API endpoints are specified.
- Commands and file paths are included where appropriate.
- Failure scenarios are documented.
- Recovery strategies are documented.
- Layout is clean and easy to read.

Export formats:

- cli-self-update.xmind
- cli-self-update.pdf
- cli-self-update.png